# Homework-3 

## Use the Titanic Dataset to Practice Data Wrangling Tasks
### Part 1: Import libaries , load the data, inspect the data, then get the name of the variables and create a small markdown table  with variable names and types (e.g numerical or categorical ordinal, categorical nominal etc.)

### Part 2: 
    - Impute missing values for "Embarked" with the mode (most frequent value)
    - Impute missing value for Age using two following strategies
        - A) Simple mean imputation (fill with the overall mean age)
        - B) Group-wise median imputation: for each combination of Sex and Pclass compute the median age. Impute missing age using the median age of passengers sex and Pclass group

### Part 3: Create a FamilySize variable (FamilySize = SibSp + Parch +1) and compute:
    - Minimum, Maximum, and mean FamilySize
    - How many passengers were traveling alone
    
### Part 4:  Create a filtered dataframe of all passengers:
    - Who are female
    - Who are male 
    - Who are under 16 years old   
    - Compute the survival rate for the above filtered data
    - Compute survival rate by Sex and Pclass (use groupby and mean) and present the result in a markdown table (rows: sex, columns: Pclass) or using print formatting.
    - Interpret your result in 2-3 sentences
     
    
### Part 5: Compute the average fare for each Pclass (Fare values are per tickets not per person (there could be more than 1 person on a ticket) so make sure to account for that. 

### Part I

In [13]:
import pandas as pd
df = pd.read_csv('titanic.csv')
df.rename(columns={ "PassengerId":"PId","Parch":"ParCh", "Survived":"Surv", "Pclass":"PClass"}, inplace= True)
df.head()

,PId,Surv,PClass,Name,Sex,Age,SibSp,ParCh,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


|Variable|Categorical Ordinal|Categorical Nominal|
|:-------|:-----------------:|:-----------------:|
|PId|x||
|Surv||x|
|PClass||x|
|Name||x|
|Sex||x|
|Age|x||
|SibSp|x||
|ParCh|x||
|Ticket||x|
|Fare|x||
|Cabin||x|
|Embarked||x|

### Part II

In [21]:
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
print(df['Embarked'].isna().sum())

0


In [25]:
mean_age = df['Age'].mean()
df['Age_mean'] = df['Age'].fillna(mean_age)
print(df['Age_mean'].isna().sum())

0


In [31]:
df['Age_median'] = df['Age'].fillna(
    df.groupby(['Sex', 'PClass'])['Age'].transform('median'))

### Part III

In [46]:
df['Family_size'] = df['SibSp'] + df['ParCh'] + 1
print("Minumum Family Size -", df['Family_size'].min())
print("Maximum Family Size -", df['Family_size'].max())
print("Average Family Size -", df['Family_size'].mean())


alone = (df['Family_size'] == 1).sum()
print("Solo Passengers", alone)

Minumum Family Size - 1
Maximum Family Size - 11
Average Family Size - 1.904601571268238
Solo Passengers 537


In [36]:
df.head()

,PId,Surv,PClass,Name,Sex,Age,SibSp,ParCh,Ticket,Fare,Cabin,Embarked,Age_mean,Age_median,Family_size
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,22.0,22.0,2
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,38.0,38.0,2
2,3,1,3,"Heikkinen, Miss Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,26.0,26.0,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,35.0,35.0,2
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,35.0,35.0,1


### Part IV

In [42]:
female = df[df['Sex'] == 'female']

male = df[df['Sex'] == 'male']

under16 = df[df['Age'] < 16]

print("Female -", len(female))
print("Male - ", len(male))
print("Under 16 -", len(under16))

Female - 314
Male -  577
Under 16 - 83


In [60]:
female_survived = female['Surv'].mean()*100
male_survived = male['Surv'].mean()*100
under_16_survived = under16['Surv'].mean()*100

print("Female Survivor Rate -", female_survived)
print("Male Survivor Rate -", male_survived)
print("Under 16 Survivor Rate -", under_16_survived)

Female Survivor Rate - 74.20382165605095
Male Survivor Rate - 18.890814558058924
Under 16 Survivor Rate - 59.036144578313255


In [57]:
survival_sex_class = df.groupby(['Sex', 'PClass'])['Surv'].mean()*100
print(survival_sex_class)

Sex     PClass
female  1         96.808511
        2         92.105263
        3         50.000000
male    1         36.885246
        2         15.740741
        3         13.544669
Name: Surv, dtype: float64


In [61]:
survival_table = df.groupby(['Sex', 'PClass'])['Surv'].mean().unstack()*100
survival_table.round(2)

PClass,1,2,3
Sex,,,
female,96.81,92.11,50.00
male,36.89,15.74,13.54


Survivor rates varied across males and females, as well as between passenger classes. Female survival rates greatly exceeded male survival rates. While higher passenger classes had higher survival rates than lower passenger classes.

### Part V

In [64]:
distinct_tickets = df.drop_duplicates(subset = 'Ticket')

avg_fare_by_class = distinct_tickets.groupby('PClass')['Fare'].mean()
print(avg_fare_by_class.round(2))

PClass
1    64.14
2    17.51
3    10.08
Name: Fare, dtype: float64
